In [0]:
tables = [
    "workspace.external_data.beroe_category_news",
    "workspace.external_data.beroe_market_analysis",
    "workspace.external_data.beroe_market_cost_structure",
    "workspace.external_data.beroe_market_price_history",
    "workspace.external_data.beroe_market_price"
]

for table in tables:
    spark.sql(f"DELETE FROM {table}")

In [0]:
%pip install openpyxl

%restart_python

import pandas as pd
import ast
from pyspark.sql.types import *
from pyspark.sql.functions import col

# Path to the Excel file
excel_path = "/Volumes/workspace/external_data/volume/beroe_datahub_all_data_v2.xlsx"

# Mapping of Excel sheet names to table names
sheet_to_table_mapping = {
    "Category News": "workspace.external_data.beroe_category_news",
    "Market Analysis": "workspace.external_data.beroe_market_analysis",
    "Market Cost Structure": "workspace.external_data.beroe_market_cost_structure",
    "Market Price History": "workspace.external_data.beroe_market_price_history",
    "Market Price": "workspace.external_data.beroe_market_price"
}

# Columns that should be arrays (per table)
array_columns = {
    "workspace.external_data.beroe_category_news": ["client_filters", "locations"],
    "workspace.external_data.beroe_market_analysis": [],
    "workspace.external_data.beroe_market_cost_structure": [],
    "workspace.external_data.beroe_market_price_history": [],
    "workspace.external_data.beroe_market_price": ["labor_utilities", "macroeconomic_factors"]
}

def convert_string_to_list(value):
    """Convert string representation of list to actual list"""
    if pd.isna(value) or value == "" or value is None:
        return None
    if isinstance(value, list):
        return value
    try:
        # Try to parse as Python literal
        return ast.literal_eval(str(value))
    except (ValueError, SyntaxError):
        # If it fails, return as single-item list
        return [str(value)]

# Read all sheet names from the Excel file
xl_file = pd.ExcelFile(excel_path)
print(f"Available sheets in Excel: {xl_file.sheet_names}\n")

# Process each sheet and upload to corresponding table
for sheet_name, table_name in sheet_to_table_mapping.items():
    print(f"Processing sheet: {sheet_name}...")
    
    try:
        # Read the Excel sheet into pandas DataFrame
        pdf = pd.read_excel(excel_path, sheet_name=sheet_name)
        
        # Store original row count before processing
        original_rows = len(pdf)
        original_cols = len(pdf.columns)
        
        # Convert column names to lowercase and replace spaces with underscores
        pdf.columns = [col.lower().replace(' ', '_').replace('/', '_') for col in pdf.columns]
        
        print(f"  - Rows: {original_rows}")
        print(f"  - Columns: {original_cols}")
        
        # Convert array columns from strings to lists
        array_cols_for_table = array_columns.get(table_name, [])
        for array_col in array_cols_for_table:
            if array_col in pdf.columns:
                pdf[array_col] = pdf[array_col].apply(convert_string_to_list)
                print(f"  - Converted {array_col} to array type")
        
        # Convert pandas DataFrame to Spark DataFrame
        spark_df = spark.createDataFrame(pdf)
        
        # Write to the target table with schema merging enabled
        spark_df.write \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .option("mergeSchema", "true") \
            .saveAsTable(table_name)
        
        # Verify the upload
        count = spark.table(table_name).count()
        print(f"  ✓ Uploaded {count} records to {table_name}\n")
        
    except Exception as e:
        print(f"  ✗ Error processing {sheet_name}: {str(e)}")
        import traceback
        print(f"  Traceback: {traceback.format_exc()}\n")

print("\n" + "="*70)
print("Upload process completed!")
print("="*70)